In [ ]:
!pip install langchain chromadb tiktoken pypdf langchain_openai langchain-community langchain-chroma langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.7 MB/s eta 0:00:00


In [ ]:
import google.generativeai as genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import CharacterTextSplitter
from google.colab import userdata

In [ ]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [ ]:
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# List available models to find a suitable embedding model
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

# The current model 'models/embedding-001' caused an error.
# You can choose one of the printed models above.
# For example, 'models/embedding-002' or 'models/text-embedding-004'.
# Update the model name below with a valid one.
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY),
    persist_directory="chroma_db",
    collection_name = 'sample'
)

models/gemini-embedding-001
models/gemini-embedding-2-preview


In [ ]:
# add documents
docs = [doc1, doc2, doc3, doc4, doc5]
vector_store.add_documents(docs)

['347e654d-fe8f-4d05-8146-e0762fdbfdf8',
 'bfe31259-591f-4f7e-9942-8613940dbd6f',
 'dff8cb40-3dbb-4f8b-b0b4-c9829213ec6d',
 '2a093207-2ad4-4d2a-bd9e-9a967f804d09',
 '0d0b6375-28db-45dc-9fde-6bb6a04e8ac8']

In [ ]:
# view documents

vector_store.get(include=['embeddings', 'documents', 'metadatas'])

{'ids': ['347e654d-fe8f-4d05-8146-e0762fdbfdf8',
  'bfe31259-591f-4f7e-9942-8613940dbd6f',
  'dff8cb40-3dbb-4f8b-b0b4-c9829213ec6d',
  '2a093207-2ad4-4d2a-bd9e-9a967f804d09',
  '0d0b6375-28db-45dc-9fde-6bb6a04e8ac8'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [ ]:
# search documents
vector_store.similarity_search(
    query="Who is Virat Kohli?",
    k=2
)

[Document(id='347e654d-fe8f-4d05-8146-e0762fdbfdf8', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
 Document(id='bfe31259-591f-4f7e-9942-8613940dbd6f', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [ ]:
# search with similarity score
vector_store.similarity_search_with_score(
    query="Who among these are a bowler?",
    k=2
)

[(Document(id='2a093207-2ad4-4d2a-bd9e-9a967f804d09', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6406819820404053),
 (Document(id='0d0b6375-28db-45dc-9fde-6bb6a04e8ac8', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.660536527633667)]

In [ ]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query=" ",
    filter={"team": "Mumbai Indians"},
    k=2
)

[(Document(id='2a093207-2ad4-4d2a-bd9e-9a967f804d09', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.7527800798416138),
 (Document(id='bfe31259-591f-4f7e-9942-8613940dbd6f', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  0.7763764262199402)]

In [ ]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [ ]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['347e654d-fe8f-4d05-8146-e0762fdbfdf8',
  'bfe31259-591f-4f7e-9942-8613940dbd6f',
  'dff8cb40-3dbb-4f8b-b0b4-c9829213ec6d',
  '2a093207-2ad4-4d2a-bd9e-9a967f804d09',
  '0d0b6375-28db-45dc-9fde-6bb6a04e8ac8'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [ ]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [ ]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['347e654d-fe8f-4d05-8146-e0762fdbfdf8',
  'bfe31259-591f-4f7e-9942-8613940dbd6f',
  'dff8cb40-3dbb-4f8b-b0b4-c9829213ec6d',
  '2a093207-2ad4-4d2a-bd9e-9a967f804d09',
  '0d0b6375-28db-45dc-9fde-6bb6a04e8ac8'],
 'embeddings': array([[-0.00982054,  0.02545763,  0.02402782, ...,  0.01414876,
         -0.01560954, -0.00266117],
        [-0.01989721,  0.01092714,  0.01730603, ...,  0.00973945,
         -0.0176257 , -0.00460104],
        [-0.01358705, -0.00359449,  0.01163961, ...,  0.01149882,
         -0.02098301,  0.00373549],
        [-0.01261678, -0.00227585,  0.00470995, ..., -0.00144414,
          0.00646786, -0.00529741],
        [-0.01413488, -0.02750698,  0.01302837, ...,  0.00987475,
         -0.00950909, -0.00272136]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca